<a href="https://colab.research.google.com/github/ravi-0309/Dynamic-Response/blob/main/Response_of_a_Truss_Tower_due_to_Ground_Motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STRUCTURE DESCRIPTION


*   **Type of Structure:** 3D Steel Truss
*   **Nodes:** 21 nodes
*   **Base nodes (0–3):** 2m × 2m square at ground level (Z = 0). All translational DOFs are restrained at base nodes, rotations are free
*   **Top node (20):** Coordinates (0, 0, 10) m
*   **Bars:** 68 members connecting nodes
*   **Height:** 10m (tapered design with intermediate nodes at Z = 2m, 4m, 6m, 8m)
*   **Degree of Static Indeterminacy** = m + r - 3j = 68 + 12 - 3*21 = 17 > 0 (Stable)
*   **E =** 200,000 MPa, **A =** 100 mm2 , **Fe250**, **Material Density =** 7850 kg/m3

# MATRIX FORMATION

## Shape Function for a 2-Noded Bar Element  
$$
[N] = \begin{bmatrix}
1- \frac{x}{L} & \frac{x}{L}
\end{bmatrix}
$$

## Stiffness Matrix  
### 1. In Local Coordinates
$$ k = \int_{0}^{V} [B]^T [E] [B]\, dV $$  
$$ where, \ Strain-Displacement \ Matrix \quad [B] = \frac{d[N]}{dx} =
\begin{bmatrix}
\frac{-1}{L} & \frac{1}{L}
\end{bmatrix}
$$  
$$ k = \int_{0}^{V}
\begin{bmatrix}
\frac{-1}{L} & \frac{1}{L}
\end{bmatrix}^T
[E]
\begin{bmatrix}
\frac{-1}{L} & \frac{1}{L}
\end{bmatrix}dV
$$  
$$
k = \frac {AE}{L}\begin{bmatrix}
1 & -1 \\
-1 & 1
\end{bmatrix}
$$

### 2. In Global Coordinates
$$
Transformaton \ Matrix, \ = \ [T] =
\begin{bmatrix}
l & m & n & 0 & 0 & 0 \\
0 & 0 & 0 & l & m & n
\end{bmatrix}
$$  
$$
where, \ l, \ m, \ n \ are \ direction \ cosines \ of \ the \ element \ in \ x, \ y, \ z \ directions
$$  
$$
K = [T]^T. [k]. [T]
$$

## Mass Matrix
$$
[N]=\begin{bmatrix}
1 - \frac {x}{L} & 0 & 0 & 0 & 0 & 0 \\
0 & \frac {x}{L} & 0 & 0 & 0 & 0 \\
0 & 0 & 1 - \frac {x}{L} & 0 & 0 & 0 \\
0 & 0 & 0 & \frac {x}{L} & 0 & 0 \\
0 & 0 & 0 & 0 & 1 - \frac {x}{L} & 0 \\
0 & 0 & 0 & 0 & 0 & \frac {x}{L}
\end{bmatrix}
$$  
$$
M = \rho.\int_{0}^{L}[N]^TdV
$$  
$$
[M] = \frac {\rho AL}{2}\begin{bmatrix}
1 & 0 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 0 & 1 \\
\end{bmatrix} (Based \ on \ Lumped-Mass \ System)
$$  
$$
where \ \ \rho \ \ is \ \ in \ \ kg/m^3
$$

# ASSEMBLING AND REDUCING THE M AND K MATRICES
All the nodes have three DOFs in x, y, z directions. \
DOFs are assigned as: \
0, 1, 2 for the first node \
3, 4, 5 for the second node \
and so on..... \
\
All the element stiffness and mass matrices are assembled in global stiffness and global mass matrix based on the DOFs of the nodes of that element. \
\
Now, the global matrices are reduced by eliminating the restrained DOFs to get the final K and M matrices, order of both these matrices is (n x n), where n = number of degrees of freedom. \
\
In this case n = 21 x 3 - 4 x 3 = 51.

# EIGEN VALUE PROBLEM AND MODE SHAPES
To find the natural frequencies and Mode Shapes, we need to solve the Eigen Value Problem, \
\
$$
[K - \omega^2M]\phi = 0
$$  
$$
where \ M, \ K \ are \ Reduced \ Mass \ and \ Stiffness \ Matrices
$$  
$$
\omega \ is \ Natural \ Frequency, \ and \ \phi \ is \ Mode \ Shape
$$

In [ ]:
# MODE SHAPES AND RESPONSE OF A TRUSS TOWER DUE TO ELCENTRO GROUND MOTION

import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from scipy.linalg import eigh
from IPython.display import HTML

# TRUSS CLASS
class Truss:
  def __init__(self, young_modulus, density, node, bar):
    self.young_modulus = young_modulus
    self.density = density
    self.node = node.astype(float)
    self.bar = bar.astype(int)

    # Truss variables
    self.dof = 3
    self.load = np.zeros_like(node)
    self.support = np.ones_like(node).astype(int)
    self.section = np.ones(len(bar))

  def analysis(self):
    nn = len(self.node)
    ne = len(self.bar)
    n_dof = self.dof * nn
    delta = self.node[self.bar[:, 1], :] - self.node[self.bar[:, 0], :]
    length = np.sqrt((delta ** 2).sum(axis = 1))
    c = (delta.T / length).T                  # Direction Cosines
    a = np.concatenate((c, -c), axis = 1)
    ss = np.zeros([n_dof, n_dof])             # Initializing Global Stiffness Matrix
    mm = np.zeros([n_dof, n_dof])             # Initializing Global Mass Matrix
    for i in range(ne):
      aux = self.dof * self.bar[i, :]
      index = np.r_[aux[0]:aux[0] + self.dof, aux[1]:aux[1] + self.dof]    # Getting the position of Element S.M. in Global S.M.

      # Element Stiffness and lumped-mass matrix
      x = c[i, 0]
      y = c[i, 1]
      z = c[i, 2]
      # p = np.sqrt(x**2 + y**2)
      t = np.array([[x, y, z, 0, 0, 0],
                    [0, 0, 0, x, y, z]])
      k = self.young_modulus * self.section[i] / length[i] * np.array([[1, -1],
                                                                       [-1, 1]])
      m = (self.density * 1e-6) * self.section[i] * length[i] / 2 * np.array([[1, 0, 0, 0, 0, 0],
                                                                              [0, 1, 0, 0, 0, 0],
                                                                              [0, 0, 1, 0, 0, 0],
                                                                              [0, 0, 0, 1, 0, 0],
                                                                              [0, 0, 0, 0, 1, 0],
                                                                              [0, 0, 0, 0, 0, 1]])
      ke = t.T @ k @ t
      me = m
      ss[np.ix_(index, index)] += ke     # Assembling the Global Matrices
      mm[np.ix_(index, index)] += me

    # Internal forces
    free_dof = self.support.flatten().nonzero()[0]   # Extracting free dofs
    kff = ss[np.ix_(free_dof, free_dof)]
    mff = mm[np.ix_(free_dof, free_dof)]

    # Dynamic analysis
    w2, vr = eigh(kff, mff)   # EVP
    w = np.sort(np.sqrt(w2))
    self.f = w / (2*np.pi)

    # return eigenvalues, eigenvector
    return w, vr, free_dof, kff, mff

# Input
modulus_elasticity = 2e5  # N / mm^2
material_density = 7850    # kg / m^3

nodes = np.array([[0, 0, 0],  # m
                  [2, 0, 0],
                  [0, 2, 0],
                  [2, 2, 0],
                  [0.2, 0.2, 2],
                  [1.8, 0.2, 2],
                  [0.2, 1.8, 2],
                  [1.8, 1.8, 2],
                  [0.4, 0.4, 4],
                  [1.6, 0.4, 4],
                  [0.4, 1.6, 4],
                  [1.6, 1.6, 4],
                  [0.6, 0.6, 6],
                  [1.4, 0.6, 6],
                  [0.6, 1.4, 6],
                  [1.4, 1.4, 6],
                  [0.8, 0.8, 8],
                  [1.2, 0.8, 8],
                  [0.8, 1.2, 8],
                  [1.2, 1.2, 8],
                  [1, 1, 10]])

bars = np.array([[4, 5],
                 [5, 7],
                 [7, 6],
                 [6, 4],
                 [8, 9],
                 [9, 11],
                 [11, 10],
                 [10, 8],
                 [12, 13],
                 [13, 15],
                 [15, 14],
                 [14, 12],
                 [16, 17],
                 [17, 19],
                 [19, 18],
                 [18, 16],
                 [0, 5],
                 [1, 4],
                 [4, 9],
                 [5, 8],
                 [8, 13],
                 [9, 12],
                 [12, 17],
                 [13, 16],
                 [16, 20],
                 [17, 20],
                 [0, 6],
                 [2, 4],
                 [4, 10],
                 [6, 8],
                 [10, 12],
                 [8, 14],
                 [12, 18],
                 [14, 16],
                 [18, 20],
                 [2, 7],
                 [3, 6],
                 [6, 11],
                 [7, 10],
                 [11, 14],
                 [10, 15],
                 [15, 18],
                 [14, 19],
                 [19, 20],
                 [1, 7],
                 [3, 5],
                 [7, 9],
                 [5, 11],
                 [11, 13],
                 [9, 15],
                 [13, 19],
                 [15, 17],
                 [0, 4],
                 [1, 5],
                 [2, 6],
                 [3, 7],
                 [4, 8],
                 [6, 10],
                 [5, 9],
                 [7, 11],
                 [8, 12],
                 [10, 14],
                 [11, 15],
                 [9, 13],
                 [12, 16],
                 [14, 18],
                 [13, 17],
                 [15, 19]])

truss_1 = Truss(modulus_elasticity, material_density, nodes, bars)

supports = truss_1.support
supports[0, :] = 0
supports[1, :] = 0
supports[2, :] = 0
supports[3, :] = 0

truss_1.section[:] = 100  # mm^2

wn, phi, free_dofs, K, M = truss_1.analysis()





# PRINTING NATURAL FREQUENCIES AND MODE SHAPE MATRIX
fn = wn / (2*np.pi)
# print("\nNatural Frequencies (Hz): ")
# print(fn)

# Mass-normalize the eigenvectors
for i in range(len(free_dofs)):
    phi[:, i] = phi[:, i] / np.sqrt(phi[:, i].T @ M @ phi[:, i])

# print("\nPhi: ")
# print(phi)





# MODE SHAPE VISUALISATION
def animate_mode_shapes():
    num_modes = min(5, len(fn))  # Animate first 5 modes

    for mode in range(num_modes):
        actual_mode_number = mode + 1
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.set_box_aspect([1, 1, 3])
        ax.set_title(f"Mode {actual_mode_number} - Frequency: {fn[mode]:.3f} Hz", fontsize=14, pad=20)

        # Initialize lines
        original_lines = []
        deformed_lines = []
        for bar in bars:
            orig_line, = ax.plot([], [], [], color='#999999', linewidth=1.0, alpha=0.7)
            def_line, = ax.plot([], [], [], color='red', linewidth=3.0, alpha=1.0)
            original_lines.append(orig_line)
            deformed_lines.append(def_line)

        # Reconstruct full mode shape (in cartesian coordinates)
        full_mode = np.zeros(3 * len(nodes))
        full_mode[free_dofs] = phi[:, mode]

        # Automatic scaling factor
        max_disp = np.max(np.abs(full_mode))
        scale = 0.2 * np.max(np.ptp(nodes, axis=0)) / (max_disp if max_disp > 0 else 1)

        # Set axis limits
        padding = 0.1
        ax.set_xlim(nodes[:,0].min()-padding, nodes[:,0].max()+padding)
        ax.set_ylim(nodes[:,1].min()-padding, nodes[:,1].max()+padding)
        ax.set_zlim(nodes[:,2].min()-padding, nodes[:,2].max()+padding)

        ax.set_xlabel('X (m)', fontsize=10)
        ax.set_ylabel('Y (m)', fontsize=10)
        ax.set_zlabel('Z (m)', fontsize=10)
        ax.grid(True)
        ax.view_init(elev=20, azim=30)

        def init():
            # Initialize the animation
            for i, bar in enumerate(bars):
                n1, n2 = bar
                # Original structure
                x = [nodes[n1,0], nodes[n2,0]]
                y = [nodes[n1,1], nodes[n2,1]]
                z = [nodes[n1,2], nodes[n2,2]]
                original_lines[i].set_data(x, y)
                original_lines[i].set_3d_properties(z)

                # Initial deformed position
                deformed_lines[i].set_data(x, y)
                deformed_lines[i].set_3d_properties(z)

            return original_lines + deformed_lines

        def update(frame):
            time = frame / 10
            displacement_factor = np.sin(2 * np.pi * time) * scale

            for i, bar in enumerate(bars):
                n1, n2 = bar

                # Original coordinates
                x1, y1, z1 = nodes[n1]
                x2, y2, z2 = nodes[n2]

                # Add scaled modal displacement
                dx1 = displacement_factor * full_mode[3*n1]
                dy1 = displacement_factor * full_mode[3*n1+1]
                dz1 = displacement_factor * full_mode[3*n1+2]

                dx2 = displacement_factor * full_mode[3*n2]
                dy2 = displacement_factor * full_mode[3*n2+1]
                dz2 = displacement_factor * full_mode[3*n2+2]

                # Update deformed lines
                deformed_lines[i].set_data(
                    [x1 + dx1, x2 + dx2],
                    [y1 + dy1, y2 + dy2]
                )
                deformed_lines[i].set_3d_properties(
                    [z1 + dz1, z2 + dz2]
                )

            return deformed_lines

        # Create animation
        ani = FuncAnimation(
            fig, update, frames=30, init_func=init,
            interval=50, blit=True, repeat=True
        )

        plt.close()
        # print(f"\nMode {actual_mode_number} Animation - Frequency: {fn[mode]:.3f} Hz - Scaling Factor: {scale:.2f}")
        display(HTML(ani.to_jshtml()))

# Execute the animation
# print("\n=== Mode Shape Animations ===")
# animate_mode_shapes()


## NORMALIZATION OF MODE SHAPES
$$
Normalized \ \{\phi\}, = \frac {\phi}{\sqrt{\phi^T.M.\phi}}
$$

## MODAL MASS AND STIFFNESS MATRICES
$$
Modal \ Mass \ Matrix, M_d = \phi^T.M.\phi = I_{51 x 51}
$$  
$$
Modal \ Stiffness \ Matrix, \ K_d = \phi^T.K.\phi = \omega^2_n \ (Diagonal \ Matrix)
$$  

## MODAL DAMPING MATRIX
$$
Using \ Rayleigh's \ Damping \ Model
$$  
$$
Modal \ Damping \ Matrix, \ C_d = \alpha M_d + \beta K_d = 2\ \xi\ \omega_n \ \ (Diagonal \ Matrix)
$$  
$$
where \ \alpha \ and \ \beta \ are \ coefficients \ which \ can \ be \ calculated \ using \ these \ equations
$$  
$$
\begin{bmatrix}
\xi_1 \\
\xi_2
\end{bmatrix} = \begin{bmatrix}
\frac {1}{2\omega_1} & \frac {\omega_1}{2} \\
\frac {1}{2\omega_2} & \frac {\omega_2}{2} \\
\end{bmatrix} \begin{bmatrix}
\alpha \\
\beta
\end{bmatrix}
$$  
$$
where \ \ \omega_1, \ \xi_1 \ = Natural \ Frequency \ and \ Damping \ ratio \ of \ the \ first \ mode. \ Assuming \ (\xi_1 = 0.02)
$$
$$
where \ \ \omega_2, \ \xi_2 \ = Natural \ Frequency \ and \ Damping \ ratio \ of \ the \ first \ mode. \ Assuming \ (\xi_2 = 0.02)
$$

## FORCE MATRIX
$$
F_d = -\phi^T.M.\{\nu\}.\ddot{x}_g
$$  
$$
where, \ \phi = Mode \ Shape \ Matrix
$$  
$$
M = Mass \ Matrix
$$  
$$
\{\nu\} = Influence \ Vector \ (DOF x 1), \ have \ value \ 1 \ corresponding \ to \ x \ direction \ DOF, \ else \ 0
$$  
$$
\ddot{x}_g = Ground \ Acceleration
$$

# PARTICIPATION FACTORS
**Total Mass:** The total mass $M_{\text{total}}$ defines how much mass is available to participate in the system's vibrations in the direction of interest. \
\
$$
M_{Total} = \{\nu\}^T.M.\{\nu\}
$$  
**Modal Participation Factor:** The modal participation factor quantifies how much each mode contributes to the total dynamic response. \
\
$$
Modal \ Participation \ Factor, \ \Gamma_i = \phi_i.M.\{\nu\}
$$  
**Effective Modal Mass:** The Effective Modal Mass (denoted as $m_{eff,i})$ tells how much of the structure's total mass is effectively moving in the i-th mode when the structure vibrates. \
\
$$
Effective \ Modal \ Mass = m_i = \Gamma_i^2
$$  
**Moda Mass Participation:** = Modal Mass Participation tells how much of the structure's total mass is participating in a specific vibration mode when the structure vibrates. \
\
$$
Modal \ Mass \ Participation = MPF (\%) = \frac {m_{eff,i}}{M_{Total}}
$$  
**As per IS 1893 (Part-1): 2016, clause 7.7.5.2** \
The number of modes N_m to be used in analysis for earthquake shaking along a considered direction, should be such that the total sum of modal masses of these modes considered is at least 90% of the total seismic mass.

# ELcentro Ground Motion


*   **Magnitude:** 6.9
*   **Depth:** Approximately 6 km
*   **Epicenter:** Near El Centro, California
*   **Fault:** Imperial Fault, a right-lateral strike-slip fault


In [ ]:
# MODAL MATRICES
# Modal Mass and Stiffness
Md = phi.T @ M @ phi
Kd = phi.T @ K @ phi

A = np.array([
    [1/(2*wn[0]), wn[0]/2],
    [1/(2*wn[1]), wn[1]/2]])

xi_1 = 0.02
xi_2 = 0.02

xi_n = np.array([[xi_1], [xi_2]])
ab = np.linalg.inv(A) @ xi_n
C = ab[0] * M + ab[1] * K
Cd = phi.T @ C @ phi  # Modal damping matrix

# Create influence vector that only affects x-direction DOFs
ifl = np.zeros((len(free_dofs),1))
for i, dof in enumerate(free_dofs):
    if dof % 3 == 0:  # Only x-direction DOFs (0, 3, 6, ...)
        ifl[i] = -1.0
F = M @ ifl  # Effective earthquake force
F_n = phi.T @ F  # Modal forces

# Modal Damping Ratio
eta_n = np.diag(Cd) / (2 * wn)
# print("\nModal Damping Ratios:")
# print(eta_n)

# print("\nModal Mass Matrix:")
# print(Md)
# print("\nModal Stiffness Matrix:")
# print(Kd)
# print("\nModal Damping Matrix:")
# print(Cd)





# ELCENTRO GROUND MOTION VISUALISATION
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File path in Google Drive
file_path = "/content/drive/My Drive/ELcentro_Accelaration_data.txt"

# Load ELcentro Acceleration data
df = pd.read_csv(file_path, delimiter="\t", header=None, names=["Time", "Acceleration"])

# Convert columns to arrays
time = df["Time"].values
xg_ddt = 9.81 * df["Acceleration"].values





# COMPUTING PARTICIPATION FACTORS
def compute_participation(M, phi, free_dofs, direction):
    # Define the influence vector for X direction
    n_dof = len(free_dofs)
    r = np.zeros(n_dof)

    for i, dof in enumerate(free_dofs):
        if direction == 'x' and dof % 3 == 0:
            r[i] = 1.0

    # Compute total mass in the given direction
    M_total = r.T @ M @ r

    # Initialize arrays to store results
    num_modes = phi.shape[1]
    Gamma = np.zeros(num_modes)   # Modal Participation factor
    m_eff = np.zeros(num_modes)   # Mass Participation factor
    MPF = np.zeros(num_modes)     # Cummulative mass Participation factor

    # Calculate Participation factors
    for i in range(num_modes):
        phi_i = phi[:, i]
        Gamma[i] = phi_i.T @ M @ r
        m_eff[i] = Gamma[i]**2  # Since phi is mass-normalized
        MPF[i] = m_eff[i] / M_total

    # Display results
    # print(f"\n=== {direction.upper()}-Direction Participation Factors ===")

    # Find where cumulative MPF reaches 90%
    cumulative_MPF = np.cumsum(MPF)
    reached_90 = False

    for i in range(num_modes):
        if cumulative_MPF[i] > 0.9 and not reached_90:
            # print(f"  Up to Mode {i+1}: {cumulative_MPF[i]*100:.2f}% (Reached 90%)")
            reached_90 = True
            break
        # print(f"Mode {i+1}:")
        # print(f"  Participation Factor (Gamma): {Gamma[i]:.4f}")
        # print(f"  Effective Modal Mass: {m_eff[i]:.4f}")
        # print(f"  Mass Participation Factor: {MPF[i]*100:.2f}%")
        # print(f"  Cumulative MPF: {cumulative_MPF[i]*100:.2f}%\n")

    # if not reached_90:
    #     print(f"  Total Cumulative MPF: {cumulative_MPF[-1]*100:.2f}% (Did not reach 90%)")

compute_participation(M, phi, free_dofs, direction='x')




# # Plot El Centro ground motion
# plt.figure(figsize=(12, 8))
# plt.plot(time, xg_ddt, 'b-', linewidth=1.5)
# plt.xlabel('Time (s)', fontsize=12)
# plt.ylabel('Acceleration (m/s2)', fontsize=12)
# plt.title('El Centro Earthquake Ground Motion (1940)', fontsize=14)
# plt.grid(True)
# plt.show()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# NEWMARK'S BETA METHOD

After decoupling the equations, we’ll get n SDOF equations which we’ll solve using the Newmark’s Beta Method. \
\
Using the average acceleration method i.e. β = 1/4 and γ = 1/2. \
\
The solution q obtained from Newmark-Beta Method is in generalized coordinates, to change it back to cartesian coordinates, we’ll pre-multiply it with transpose of phi to get u (u is relative response w.r.t. ground). \
$$
u_r = \phi^T.q
$$

In [ ]:
# FUNCTION TO CALCULATE DYNAMIC RESPONSE USING NEWMARK'S METHOD
def newmark_beta_solver(m, k, eta, time, f_t, beta=1/4, gamma=1/2):

    dt = time[1] - time[0]
    n = len(time)
    c = 2 * eta * np.sqrt(m * k)
    x = np.zeros(n)
    v = np.zeros(n)
    a = np.zeros(n)

    # Initial acceleration
    f0 = np.interp(time[0], time, f_t)
    a[0] = (f0 - c * v[0] - k * x[0]) / m

    # Effective stiffness
    k_eff = k + gamma * c / (beta * dt) + m / (beta * dt**2)

    for i in range(1, n):
        f_eff = (
            np.interp(time[i], time, f_t) +
            m * (1 / (beta * dt**2) * x[i-1] + 1 / (beta * dt) * v[i-1] + (1 / (2 * beta) - 1) * a[i-1]) +
            c * (gamma / (beta * dt) * x[i-1] + (gamma / beta - 1) * v[i-1] + dt * (gamma / (2 * beta) - 1) * a[i-1])
        )

        x[i] = f_eff / k_eff
        v[i] = (
            gamma / (beta * dt) * (x[i] - x[i-1]) +
            (1 - gamma / beta) * v[i-1] +
            dt * (1 - gamma / (2 * beta)) * a[i-1]
        )
        a[i] = (
            1 / (beta * dt**2) * (x[i] - x[i-1]) -
            1 / (beta * dt) * v[i-1] -
            (1 / (2 * beta) - 1) * a[i-1]
        )

    return x

# Compute modal responses
num_modes = len(wn)
Z_n_1 = np.zeros((num_modes, len(time)))

for i in range(num_modes):
    Z_n_1[i, :] = newmark_beta_solver(1, wn[i]**2, eta_n[i], time, F_n[i] * xg_ddt)

# Transform back to Cartesian coordinates
X_t_1 = 10e3 * phi @ Z_n_1  # X_t is relative displacement of free DOFs w.r.t. ground

# Set higher animation embed limit (in MB)
plt.rcParams['animation.embed_limit'] = 500  # Increase to 500MB





# # PLOT THE RESPONSE OF SOME TOP STOREY DOFS
# for i in range(45, len(free_dofs)):
#     plt.figure(figsize=(15, 6))
#     plt.plot(time, X_t_1[i, :], label=f'DOF {free_dofs[i]+1}')
#     plt.xlabel('Time (s)')
#     plt.ylabel('Displacement (mm)')
#     plt.title('Newmark-Beta Method')
#     plt.legend()
#     plt.grid(True)
#     plt.show()




# MAXIMUM DISPLACEMENT ANALYSIS
max_displacement = 0
max_node = 0
max_dof = 0
max_time_idx = 0

for time_idx in range(len(time)):
    for i, dof in enumerate(free_dofs):
        current_disp = abs(X_t_1[i, time_idx])
        if current_disp > max_displacement:
            max_displacement = current_disp
            max_dof = dof
            max_time_idx = time_idx

# Determine which node this DOF belongs to
node_number = max_dof // 3  # Since each node has 3 DOFs (x,y,z)
dof_type = max_dof % 3      # 0=x, 1=y, 2=z

# Get the displacement values
actual_disp = X_t_1[np.where(free_dofs == max_dof)[0][0], max_time_idx]

# Print results
# print(f"Maximum displacement occurs at node {node_number}")
# print(f"Displacement component: {'X' if dof_type == 0 else 'Y' if dof_type == 1 else 'Z'}")
# print(f"Time of maximum displacement: {time[max_time_idx]:.2f} seconds")
# print(f"Maximum displacement value: {actual_disp:.2f} mm")

In [ ]:
def animate_3d_truss_response():
    nodes_mm = nodes * 1000  # Convert nodes to mm
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_box_aspect([1, 1, 3])

    scale = 50  # Scale factor for displacements

    ax.set_title("3D Truss Tower Response to El Centro Earthquake", fontsize=14, pad=20)

    # Initialize lines
    original_lines = []
    deformed_lines = []
    for bar in bars:
        # Original structure lines (gray)
        orig_line, = ax.plot([], [], [], color='#999999', linewidth=1.0, alpha=0.7)
        # Deformed structure lines (red)
        def_line, = ax.plot([], [], [], color='red', linewidth=3.0, alpha=1.0)
        original_lines.append(orig_line)
        deformed_lines.append(def_line)

    # Add time text annotation
    time_text = ax.text2D(0.02, 0.95, '', transform=ax.transAxes, fontsize=12,
                         bbox=dict(facecolor='white', alpha=0.8))

    # Set axis limits (convert back to meters for display)
    display_scale = 1000  # Convert mm back to meters
    padding = 0.2 * display_scale
    ax.set_xlim((nodes_mm[:,0].min()-padding)/display_scale, (nodes_mm[:,0].max()+padding)/display_scale)
    ax.set_ylim((nodes_mm[:,1].min()-padding)/display_scale, (nodes_mm[:,1].max()+padding)/display_scale)
    ax.set_zlim(0, (nodes_mm[:,2].max()+padding)/display_scale)

    ax.set_xlabel('X (m)', fontsize=10)
    ax.set_ylabel('Y (m)', fontsize=10)
    ax.set_zlabel('Z (m)', fontsize=10)
    ax.grid(True)
    ax.view_init(elev=20, azim=30)  # Set initial view angle

    def init():
        # Initialize the animation with original structure
        for i, bar in enumerate(bars):
            n1, n2 = bar
            # Original coordinates
            x = [nodes_mm[n1,0], nodes_mm[n2,0]]
            y = [nodes_mm[n1,1], nodes_mm[n2,1]]
            z = [nodes_mm[n1,2], nodes_mm[n2,2]]

            original_lines[i].set_data(np.array(x)/display_scale, np.array(y)/display_scale)
            original_lines[i].set_3d_properties(np.array(z)/display_scale)

            # Initial deformed position (same as original)
            deformed_lines[i].set_data(np.array(x)/display_scale, np.array(y)/display_scale)
            deformed_lines[i].set_3d_properties(np.array(z)/display_scale)

        # Initialize time text
        time_text.set_text('Time: 0.00 s')

        return original_lines + deformed_lines + [time_text]

    step = max(1, len(time) // 200)
    time_indices = np.arange(0, len(time), step)

    def update(frame):
        time_idx = time_indices[frame]
        current_time = time[time_idx]

        # Update time display
        time_text.set_text(f'Time: {current_time:.2f} s')

        # Get displacements at this time step
        displacements = np.zeros(3 * len(nodes_mm))
        for i, dof in enumerate(free_dofs):
            displacements[dof] = X_t_1[i, time_idx]

        # Update each bar
        for i, bar in enumerate(bars):
            n1, n2 = bar

            # Original coordinates (in mm)
            x1, y1, z1 = nodes_mm[n1]
            x2, y2, z2 = nodes_mm[n2]

            # Add scaled displacements
            dx1 = scale * displacements[3*n1]
            dy1 = scale * displacements[3*n1+1]
            dz1 = scale * displacements[3*n1+2]

            dx2 = scale * displacements[3*n2]
            dy2 = scale * displacements[3*n2+1]
            dz2 = scale * displacements[3*n2+2]

            # Update deformed lines (convert back to meters for display)
            x_coords = np.array([x1 + dx1, x2 + dx2])/display_scale
            y_coords = np.array([y1 + dy1, y2 + dy2])/display_scale
            z_coords = np.array([z1 + dz1, z2 + dz2])/display_scale

            deformed_lines[i].set_data(x_coords, y_coords)
            deformed_lines[i].set_3d_properties(z_coords)

        return deformed_lines + [time_text]

    # Create animation
    ani = FuncAnimation(
        fig, update, frames=len(time_indices), init_func=init,
        interval=20, blit=True, repeat=True
    )

    plt.close()
    display(HTML(ani.to_jshtml()))

# Run the animation
# animate_3d_truss_response()

# AXIAL FORCE CALCULATION

1. Compute element length L and Direction Cosines $\vec{c}$
2. Compute Displacement vector of the two nodes of the element $u_2 - u_1$
3. Axial Deformation = $\vec{c} . (\vec{u_2} - \vec{u_1})$.
4. Compute corresponding Axial Forces using F = EA$\Delta$ / L and store it in an array.
5. Finally, Compute the maximum Axial Forces.

In [ ]:
# Calculate axial forces in all members over time
def calculate_axial_forces(truss, X_t_1, time, free_dofs):
    nn = len(truss.node)
    ne = len(truss.bar)
    axial_forces = np.zeros((ne, len(time)))

    # Convert nodes to mm (consistent with displacement units)
    nodes_mm = truss.node * 1000

    for t in range(len(time)):
        # Reconstruct full displacement vector (in mm)
        U = np.zeros(3 * nn)
        for i, dof in enumerate(free_dofs):
            U[dof] = X_t_1[i, t]

        for e in range(ne):
            # Get element properties
            n1, n2 = truss.bar[e]
            delta = nodes_mm[n2] - nodes_mm[n1]
            L = np.sqrt((delta ** 2).sum())
            c = delta / L  # Direction cosines

            # Element displacement vector
            u1 = U[3*n1 : 3*n1+3]
            u2 = U[3*n2 : 3*n2+3]

            # Axial deformation
            axial_deformation = np.dot(c, (u2 - u1))

            # Axial force (F = EAΔL/L)
            axial_force = (truss.young_modulus * truss.section[e] * axial_deformation) / L
            axial_forces[e, t] = axial_force

    return axial_forces

# Calculate axial forces
axial_forces = calculate_axial_forces(truss_1, X_t_1, time, free_dofs)

# Find member with maximum absolute axial force
max_force = 0
max_member = 0
max_time_idx = 0

for e in range(len(bars)):
    for t in range(len(time)):
        current_force = abs(axial_forces[e, t])
        if current_force > max_force:
            max_force = current_force
            max_member = e
            max_time_idx = t

# Get the actual force value
actual_force = axial_forces[max_member, max_time_idx]

# Print results
# print("\n=== Maximum Axial Force Analysis ===")
# print(f"Member with highest axial force: Bar {max_member} (between nodes {bars[max_member, 0]} and {bars[max_member, 1]})")
# print(f"Time of maximum force: {time[max_time_idx]:.2f} seconds")
# print(f"Maximum axial force: {actual_force:.2f} N")

# Print member properties
n1, n2 = bars[max_member]
# print(f"\nMember properties:")
# print(f"Length: {np.linalg.norm(nodes[n2] - nodes[n1]):.2f} m")
# print(f"Cross-section: {truss_1.section[max_member]} mm²")
# print(f"Material: E = {truss_1.young_modulus:.2e} N/mm², ρ = {truss_1.density} kg/m³")

# Print top 5 members with highest forces
# print("\nTop 5 members with highest axial forces:")
max_forces = np.max(np.abs(axial_forces), axis=1)
top_members = np.argsort(-max_forces)[:5]
# for i, member in enumerate(top_members):
#     n1, n2 = bars[member]
#     print(f"{i+1}. Bar {member} (Nodes {n1}-{n2}): {max_forces[member]:.2f} N")

# CONCLUSION

*   The maximum deflection occurs at the top-most node in the direction of ground motion which is equal to 10.68 mm.
*   The allowable sway limit as per IS 800:2007 is H/250 i.e. 40 mm. Hence the deflection is under limit.
*   The members experiencing largest axial force are the bottom most ones, maximum axial force is 3998 N, and the strength of the Fe250 bar is = 250 * 100 = 25000 N
*   Factor of Safety = 25000 / 3998 = 6.25